In [ ]:
# Install all required libraries including pyvene for steering inference
!pip install -q torch transformers accelerate bitsandbytes huggingface_hub pyvene

In [ ]:
from huggingface_hub import login
login(token="YOUR_TOKEN_HERE")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/samiagul430-lang/sycophancy
%cd sycophancy

In [ ]:
!pip install torch transformers accelerate bitsandbytes pyvene datasets jsonlines pandas tqdm huggingface_hub

In [ ]:
!pip install nnsight

In [ ]:
!pwd && ls -la && grep -rn "from_pretrained" .

In [ ]:
!sed -n '1,48p' utils.py && echo "=== EXTRACT ===" && sed -n '48,70p' extension/extract_activations.py && echo "=== PROBE ===" && ls -la probe

In [ ]:
# Read the existing utils.py file
with open("utils.py", "r") as f:
  code = f.read()

# Define original and replacement strings for 8-bit quantization
old_import = (
    "from transformers import AutoProcessor, AutoModelForCausalLM,"
    " AutoTokenizer, Gemma3ForConditionalGeneration"
)
new_import = (
    "from transformers import AutoProcessor, AutoModelForCausalLM,"
    " AutoTokenizer, Gemma3ForConditionalGeneration, BitsAndBytesConfig"
)

old_load = (
    'model ='
    ' Gemma3ForConditionalGeneration.from_pretrained("google/gemma-3-4b-it",'
    " device_map=device)"
)
new_load = (
    "quant_config = BitsAndBytesConfig(load_in_8bit=True)\n        model ="
    ' Gemma3ForConditionalGeneration.from_pretrained("google/gemma-3-4b-it",'
    " quantization_config=quant_config, device_map=device)"
)

# Verify patterns exist and replace them
assert old_import in code and old_load in code, "Pattern mismatch in utils.py!"
code = code.replace(old_import, new_import).replace(old_load, new_load)

# Save the updated configuration back to utils.py
with open("utils.py", "w") as f:
  f.write(code)

print("SUCCESS: utils.py updated with 8-bit configuration!\n")

In [ ]:
%%bash
# Display README instructions and CLI arguments for the pipeline scripts
cat README.md
echo "=== ARGUMENTS ==="
grep -rn "add_argument" probe/

In [ ]:
%%bash
# Find all occurrences of language_model across the repo and inspect extract_activation.py
grep -rn "language_model" .
echo "=== EXTRACT_ACTIVATION ==="
sed -n '1,65p' probe/extract_activation.py

In [ ]:
# Fix Gemma-3 layer path in probe/extract_activation.py
with open("probe/extract_activation.py", "r") as f:
  extract_code = f.read()

extract_code = extract_code.replace(
    "model.language_model.model.layers", "model.model.language_model.layers"
).replace("model.language_model.layers", "model.model.language_model.layers")

with open("probe/extract_activation.py", "w") as f:
  f.write(extract_code)

# Fix Gemma-3 component path in inference_mha.py for pyvene steering
with open("inference_mha.py", "r") as f:
  infer_code = f.read()

infer_code = infer_code.replace(
    'f"language_model.layers[', 'f"model.language_model.layers['
)

with open("inference_mha.py", "w") as f:
  f.write(infer_code)

print("SUCCESS: Gemma-3 layer paths fixed in both training and inference files!")

In [ ]:
%%bash
# Run activation extraction and linear probe training with live unbuffered output
PYTHONUNBUFFERED=1 PYTHONPATH=. python3 probe/train.py --model_id gemma-3 --activation_type mha --concept sycophancy

In [ ]:
import shutil
import os

# Copy trained probes directly to Google Drive using Python to ensure safety
source_dir = 'trained_probe_sycophancy'
dest_dir = '/content/drive/MyDrive/sycophancy_backup/trained_probe_sycophancy'

if os.path.exists(source_dir):
    shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True)
    print('SUCCESS: All trained probes successfully backed up to Google Drive!')
else:
    print('Error: Source directory not found.')

In [ ]:
%%bash
# Drive se trained probes ko wapas Colab ke local folder mein copy karein
mkdir -p trained_probe_sycophancy
cp -r /content/drive/MyDrive/sycophancy_backup/trained_probe_sycophancy/* trained_probe_sycophancy/
echo "SUCCESS: Probes successfully loaded from Google Drive! No training needed."

In [ ]:
%%bash
PYTHONPATH=. python3 probe/compute_proj_std.py --model_id gemma-3 --activation_type mha --concept sycophancy --probe_type linear

In [ ]:
%%bash
PYTHONUNBUFFERED=1 PYTHONPATH=. python3 probe/compute_proj_std.py --model_id gemma-3 --activation_type mha --concept sycophancy --probe_type linear

In [ ]:
import os
import torch
from transformers import BitsAndBytesConfig

# Set environment variables
os.environ['PYTHONPATH'] = '.'
os.environ['PYTHONUNBUFFERED'] = '1'

# 1. Configure 4-bit quantization (NF4) for fast and clean execution
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("🚀 Starting 4-bit Inference and Evaluation...")

# 2. Run the inference script with 4-bit settings
!python3 inference_mha.py --model_id gemma-3 --load_in_4bit

In [ ]:
import torch
import pickle
import os

print("✔ Environment active. Checking trained probe files...")

# Check if accuracy file already exists from previous runs
probe_path = "probe/trained_probe_sycophancy/gemma-3/linear_accuracies_dict_mha.pkl"
if os.path.exists(probe_path):
    with open(probe_path, "rb") as f:
        accuracies = pickle.load(f)
    print("🎯 SUCCESS! Found existing probe accuracies result:")
    print(accuracies)
else:
    print("⚠️ Probe accuracy file is generating or located elsewhere. Let's check the folder.")

In [ ]:
import pickle
import pandas as pd
import os

# Load accuracies dictionary
probe_path = "probe/trained_probe_sycophancy/gemma-3/linear_accuracies_dict_mha.pkl"

if os.path.exists(probe_path):
    with open(probe_path, "rb") as f:
        accuracies = pickle.load(f)

    # Format into a Layer x Head matrix
    df_dict = {}
    for key, acc in accuracies.items():
        layer, head = key.split('_')
        layer, head = int(layer), int(head)
        if layer not in df_dict:
            df_dict[layer] = {}
        df_dict[layer][f'Head {head}'] = acc

    df = pd.DataFrame.from_dict(df_dict, orient='index').sort_index()

    # Highlight the maximum accuracy in the entire table
    styled_df = df.style.highlight_max(axis=None, color='lightgreen', props='background-color: yellow; color: black; font-weight: bold;')

    print("🎯 Top Performing Probe Table (Layer 16, Head 0 Highlighted):")
    display(styled_df)
else:
    print("❌ Accuracy file not found!")

In [ ]:
import torch
import os

layer_idx = 16
head_idx = 0
probe_file = f"probe/trained_probe_sycophancy/gemma-3/linear_std_mha_{layer_idx}_{head_idx}.pt"

if os.path.exists(probe_file):
    data = torch.load(probe_file)
    print(f"Loaded object type: {type(data)}")

    if isinstance(data, dict):
        print("Keys inside dict:", data.keys())
        # Let's find the actual weight tensor
        for k, v in data.items():
            if isinstance(v, torch.Tensor):
                print(f"Key '{k}' has shape: {v.shape}")
    elif isinstance(data, torch.Tensor):
        print(f"Direct tensor shape: {data.shape}")
else:
    print("Probe file not found.")

In [ ]:
import os

files = os.listdir("probe/trained_probe_sycophancy/gemma-3/")
print("Files in directory:", [f for f in files if f.endswith('.pt') or f.endswith('.pkl')])

In [ ]:
import torch

# Load the exact file for Layer 16, Head 0
probe_file = "probe/trained_probe_sycophancy/gemma-3/linear_std_mha_16_0.pt"
checkpoint = torch.load(probe_file, map_location="cpu")

print("Type of loaded object:", type(checkpoint))

# Check if it's a state dictionary or direct weights
if isinstance(checkpoint, dict):
    print("Keys in checkpoint:", checkpoint.keys())
    # Usually linear weights are stored under 'weight' or similar keys
    for k, v in checkpoint.items():
        if isinstance(v, torch.Tensor):
            print(f"Key: {k} | Shape: {v.shape}")
elif isinstance(checkpoint, torch.Tensor):
    print("Tensor shape:", checkpoint.shape)

In [ ]:
import torch

# Test a few other top-performing heads in Layer 16
for h in [3, 4]:
    f_path = f"probe/trained_probe_sycophancy/gemma-3/linear_std_mha_16_{h}.pt"
    chk = torch.load(f_path, map_location="cpu")
    print(f"Head {h} - Type: {type(chk)}, Shape: {chk.shape if isinstance(chk, torch.Tensor) else 'Dict'}")

### 📊 Sycophancy Probe Results (Quantized Gemma-3)
* **Status:** Quantization & Layer-wise Sweep Completed
* **Top Performing Head:** Layer 16, Head 0
* **Highest Accuracy:** **87.38%**
* **Key Finding:** Sycophancy representations are robustly encoded in the middle-to-late network layers of the model.